# Human validation: build the blind labeling sample

Rebuilds the exact val split the pipeline scored, auto-labels every val frame with the same
YOLOv8x + footprint procedure, and samples ~100 frames for two annotators to label blind in
`label_tool.html`.

**Attach:** `janahussien90/rgb3thermals-full` and `janahussien90/homography-mats`. **Accelerator:** GPU (T4).
Set `SMOKE_TEST = True` in the first cell for a 2-minute end-to-end check before the real run.

**Outputs** (in `/kaggle/working`, download from the Output tab):

| File | Who gets it |
|---|---|
| `hv_sample.zip` | Annotators. It holds `manifest.json` + RGB frames with **no labels**. Unzip it and open it in `label_tool.html`. |
| `hv_key.zip` | **Analysis only.** It holds the auto-labels and YOLO boxes for the sample, plus auto-labels for all val frames (for prevalence weighting). Don't open it until both annotators have exported. |

This notebook deliberately never displays auto-labels or YOLO boxes, because whoever runs it is also an annotator.

In [ ]:
!pip install -q ultralytics

import json, re, random, shutil, warnings, zipfile
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.path import Path as MplPath
from ultralytics import YOLO

# ---- must match the pipeline (cluster_pipeline/scripts/00_setup_and_labels.py) ----
SEED = 42
VAL_FRAC = 0.2
N_SPLIT_BLOCKS = 10
MAX_RGB_GAP_MS = 150
YOLO_WEIGHTS = "yolov8x.pt"
COCO_PERSON_CLASS_ID = 0
SENSOR_W, SENSOR_H = 32, 24
ACTIVE_CAMERAS = [2, 5, 4]                  # left, middle, right
DIRS = ["L", "M", "R"]
CAM_TO_DIR = dict(zip(ACTIVE_CAMERAS, DIRS))
PANORAMA_REFERENCE_CAM = 5

# ---- sample design ----
N_PERSON_FRAMES = 35      # frames with >=1 direction auto-labeled person (rotated over L/M/R)
N_OTHER_FRAMES = 35       # frames with >=1 direction auto-labeled other  (rotated over L/M/R)
N_RANDOM_FRAMES = 30      # any remaining val frame (mostly clear)
MIN_GAP_ANCHORS = 15      # no two sampled frames within this many anchors of each other in one part
MIN_PER_CLASS = 45        # warn if any class ends up below this many directions
OUT_MAX_WIDTH = 1600      # downscale RGB frames wider than this (footprints scaled to match)

# ---- smoke test: a few hundred val frames and a 15-frame sample, to check the notebook end-to-end ----
SMOKE_TEST = False
SMOKE_N_VAL = 300
if SMOKE_TEST:
    N_PERSON_FRAMES = N_OTHER_FRAMES = N_RANDOM_FRAMES = 5
    MIN_PER_CLASS = 0
    print("*** SMOKE TEST: outputs are NOT the real sample -- set SMOKE_TEST = False for the real run ***")

OUT = Path("/kaggle/working")
SAMPLE_DIR = OUT / "hv_sample"
KEY_DIR = OUT / "hv_key"
rng = random.Random(SEED)

## 1. Locate the data

In [ ]:
import os, zipfile

DATASET = "full"            # "sample" -> rgb3thermals_sample/pictureData, "full" -> rgb3thermals_full/all_runs
PICTURE_DIR_NAMES = {"sample": "pictureData", "full": "all_runs"}
CALIBRATION_NAMES = ["thermal_homography_mats", "rgb homography final"]
KAGGLE_INPUT = Path("/kaggle/input")
EXTRACT_TO = Path("/kaggle/working/rgb3thermals_extracted")   # /kaggle/input is read-only
MAX_SEARCH_DEPTH = 6


def _is_capture_dir(name):
    return bool(re.fullmatch(r"part\d+", name)) or name.startswith(("thermal-g", "rgb-g", "sync-g"))


def _walk_dirs(root, max_depth=MAX_SEARCH_DEPTH):
    stack = [(Path(root), 0)]
    while stack:
        d, depth = stack.pop()
        try:
            entries = list(os.scandir(d))
        except (PermissionError, NotADirectoryError):
            continue
        yield d, {e.name for e in entries}
        if depth < max_depth:
            for e in entries:
                if e.is_dir() and e.name != "__MACOSX" and not _is_capture_dir(e.name):
                    stack.append((Path(e.path), depth + 1))


def find_base(names, search_root=KAGGLE_INPUT, hint=""):
    """The single folder under search_root that contains every name in `names` (as a folder or .zip)."""
    complete = sorted(d for d, entries in _walk_dirs(search_root)
                      if all(n in entries or f"{n}.zip" in entries for n in names))
    if not complete:
        raise FileNotFoundError(f"No folder under {search_root} contains all of {names} (folder or .zip). {hint}")
    base = min(complete, key=lambda d: len(d.parts))
    others = [d for d in complete if d != base and base not in d.parents]
    if others:
        raise ValueError(f"Multiple candidate folders for {names}: {[str(base)] + [str(d) for d in others]}")
    return base


def ensure_extracted(base, name):
    """base/<name> if it is a folder, else extracts base/<name>.zip into EXTRACT_TO/<name> (skipping macOS junk)."""
    for candidate in [base / name, EXTRACT_TO / name]:
        if candidate.is_dir():
            print(f"using folder: {candidate}")
            return candidate
    zip_path = base / f"{name}.zip"
    if not zip_path.is_file():
        raise FileNotFoundError(f"Neither folder '{name}' nor '{name}.zip' found in {base}")
    EXTRACT_TO.mkdir(parents=True, exist_ok=True)
    dest, tmp = EXTRACT_TO / name, EXTRACT_TO / f".{name}.extracting"
    if tmp.exists():
        shutil.rmtree(tmp)
    print(f"extracting {zip_path.name} -> {dest} ...")
    with zipfile.ZipFile(zip_path) as zf:
        members = [m for m in zf.namelist() if not m.startswith("__MACOSX/") and not Path(m).name.startswith("._")]
        zf.extractall(tmp, members=members)
    tmp.rename(dest)
    return dest


CALIB_BASE = find_base(CALIBRATION_NAMES, hint="Is janahussien90/homography-mats attached?")
PICTURE_BASE = find_base([PICTURE_DIR_NAMES[DATASET]], hint="Is janahussien90/rgb3thermals-full attached?")
DATA_ROOT = ensure_extracted(PICTURE_BASE, PICTURE_DIR_NAMES[DATASET])
HOMOGRAPHY_DIR = ensure_extracted(CALIB_BASE, "thermal_homography_mats")
RGB_HOMOGRAPHY_DIR = ensure_extracted(CALIB_BASE, "rgb homography final")


def _part_sort_key(p):
    m = re.search(r"(\d+)$", p.name)
    return int(m.group(1)) if m else 0


def _find_one(root, prefix):
    matches = sorted(p for p in Path(root).glob(f"{prefix}-g*") if p.is_dir())
    if len(matches) > 1:
        warnings.warn(f"{Path(root).name}: several '{prefix}-g*' folders, using {matches[0].name}")
    return matches[0] if matches else None


def _find_part_dirs(root, max_depth=3):
    """partN folders may sit directly in DATA_ROOT or a level or two deeper (zip wrapper folders)."""
    found, stack = [], [(Path(root), 0)]
    while stack:
        d, depth = stack.pop()
        for e in os.scandir(d):
            if not e.is_dir() or e.name == "__MACOSX":
                continue
            if re.fullmatch(r"part\d+", e.name):
                found.append(Path(e.path))
            elif depth < max_depth:
                stack.append((Path(e.path), depth + 1))
    return found


# Same order as the pipeline (numeric part order) -- the split depends on it.
CAPTURE_PARTS, SKIPPED_PARTS = {}, {}
for root in sorted(_find_part_dirs(DATA_ROOT), key=_part_sort_key):
    paths = {"thermal_dir": _find_one(root, "thermal"), "rgb_dir": _find_one(root, "rgb"), "sync_dir": _find_one(root, "sync")}
    missing = [k for k, v in paths.items() if v is None]
    if not missing and not (paths["sync_dir"] / "sync.csv").is_file():
        missing = ["sync.csv"]
    if missing:
        SKIPPED_PARTS[root.name] = missing
        continue
    paths["sync_csv"] = paths.pop("sync_dir") / "sync.csv"
    CAPTURE_PARTS[root.name] = paths
if SKIPPED_PARTS:
    warnings.warn(f"Skipping incomplete parts (the cluster pipeline would have crashed on these, "
                  f"so the split may differ from the paper's): {SKIPPED_PARTS}")
print("captures:   ", DATA_ROOT)
print("thermal H:  ", HOMOGRAPHY_DIR)
print("rgb H:      ", RGB_HOMOGRAPHY_DIR)
print(f"{len(CAPTURE_PARTS)} parts: {list(CAPTURE_PARTS)}")

## 2. Rebuild the pipeline's train/val split

This is a line-for-line copy of `load_all_sync_rows` + `blocked_train_val_split`. All sessions are combined, shuffled with
`random.Random(42)`, and the last 20% of each of 10 blocks becomes val. If the part list or row
counts differ from the cluster's `all_runs - Copy`, the split will differ too, so check the totals below.

In [ ]:
all_rows = []
for part, paths in CAPTURE_PARTS.items():
    rows = pd.read_csv(paths["sync_csv"]).to_dict("records")
    for i, row in enumerate(rows):
        row["_part"] = part
        row.setdefault("anchor_index", i)
    all_rows.extend(rows)
random.Random(SEED).shuffle(all_rows)

def blocked_train_val_split(rows, val_frac=VAL_FRAC, n_blocks=N_SPLIT_BLOCKS):
    n = len(rows)
    block_size = max(1, n // n_blocks)
    train_rows, val_rows = [], []
    for start in range(0, n, block_size):
        block = rows[start:start + block_size]
        n_val = max(1, int(len(block) * val_frac)) if len(block) > 1 else 0
        if n_val == 0:
            train_rows.extend(block)
        else:
            train_rows.extend(block[:-n_val])
            val_rows.extend(block[-n_val:])
    if not val_rows:
        val_rows = train_rows[-1:]
    return train_rows, val_rows

def rgb_gap_ok(row, max_gap_ms=MAX_RGB_GAP_MS):
    files = row.get("rgb_files")
    if not isinstance(files, str) or not files:
        return False
    try:
        return 0 <= float(row.get("rgb_gap_ms", -1)) <= max_gap_ms
    except (ValueError, TypeError):
        return False

train_rows, val_rows = blocked_train_val_split(all_rows)
val_usable = [r for r in val_rows if rgb_gap_ok(r)]
if SMOKE_TEST:
    val_usable = val_usable[:SMOKE_N_VAL]
print(f"total sync rows: {len(all_rows)}   train: {len(train_rows)}   val: {len(val_rows)}")
print(f"val rows with a usable RGB frame (labelable): {len(val_usable)}")
print("(the paper says 15,500 frames were used for training -- train here should be close to that)")
print(pd.Series([r["_part"] for r in val_usable]).value_counts().sort_index().to_string())

## 3. Geometry: footprints in the panorama and in RGB pixels

In [ ]:
CAM_NAME_RE = re.compile(r"cam(\d+)")

def _corners(w, h, T):
    c = np.array([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]], float)
    out = (T @ np.hstack([c, np.ones((4, 1))]).T).T
    return out[:, :2] / out[:, 2:3]

homographies = []
for path in sorted(HOMOGRAPHY_DIR.rglob("*homography*.json")):
    if path.name.startswith("._") or "__MACOSX" in path.parts:
        continue
    d = json.loads(path.read_text())
    a, b = CAM_NAME_RE.match(d["cam_a"]), CAM_NAME_RE.match(d["cam_b"])
    homographies.append({"cam_a": int(a.group(1)), "cam_b": int(b.group(1)), "H": np.array(d["H"], float)})

graph = {c: [] for c in ACTIVE_CAMERAS}
for e in homographies:
    if e["cam_a"] in graph and e["cam_b"] in graph:
        graph[e["cam_a"]].append((e["cam_b"], np.linalg.inv(e["H"])))
        graph[e["cam_b"]].append((e["cam_a"], e["H"]))
CAM_TRANSFORMS, queue = {PANORAMA_REFERENCE_CAM: np.eye(3)}, [PANORAMA_REFERENCE_CAM]
while queue:
    cur = queue.pop(0)
    for nb, T_nb in graph[cur]:
        if nb not in CAM_TRANSFORMS:
            CAM_TRANSFORMS[nb] = CAM_TRANSFORMS[cur] @ T_nb
            queue.append(nb)
assert set(ACTIVE_CAMERAS) <= set(CAM_TRANSFORMS), f"unlinked cameras: {set(ACTIVE_CAMERAS) - set(CAM_TRANSFORMS)}"

pts = np.vstack([_corners(SENSOR_W, SENSOR_H, T) for T in CAM_TRANSFORMS.values()])
min_x, min_y = np.floor(pts.min(0)).astype(int)
TRANSLATION = np.array([[1, 0, -min_x], [0, 1, -min_y], [0, 0, 1]], float)
CAM_FOOTPRINTS = {c: _corners(SENSOR_W, SENSOR_H, TRANSLATION @ T) for c, T in CAM_TRANSFORMS.items()}

RGB_CALIBRATIONS = {}
for path in sorted(RGB_HOMOGRAPHY_DIR.rglob("rgb*_homography.json")):
    if path.name.startswith("._") or "__MACOSX" in path.parts:
        continue
    d = json.loads(path.read_text())
    cam = int(CAM_NAME_RE.match(d["cam_b"]).group(1))
    RGB_CALIBRATIONS[cam] = {"H": np.array(d["H"], float), "centroid": np.array(d["quad_a"], float).mean(0)}
SUPERVISED_CAMS = sorted(set(RGB_CALIBRATIONS) & set(CAM_TRANSFORMS))
assert set(ACTIVE_CAMERAS) <= set(SUPERVISED_CAMS), f"missing RGB calibration for {set(ACTIVE_CAMERAS) - set(SUPERVISED_CAMS)}"
print("cameras with RGB supervision:", SUPERVISED_CAMS)

# RGB -> panorama for camera c is TRANSLATION @ CAM_TRANSFORMS[c] @ H_rgb[c]; its inverse maps the
# footprint polygon back into RGB pixels so annotators see the same region the labeler used.
def footprint_in_rgb(cam, n_per_edge=8):
    M_inv = np.linalg.inv(TRANSLATION @ CAM_TRANSFORMS[cam] @ RGB_CALIBRATIONS[cam]["H"])
    poly = CAM_FOOTPRINTS[cam]
    dense = np.vstack([np.linspace(poly[i], poly[(i + 1) % 4], n_per_edge, endpoint=False) for i in range(4)])
    out = (M_inv @ np.hstack([dense, np.ones((len(dense), 1))]).T).T
    return out[:, :2] / out[:, 2:3]

RGB_FOOTPRINTS = {CAM_TO_DIR[c]: footprint_in_rgb(c) for c in ACTIVE_CAMERAS}
for d, p in RGB_FOOTPRINTS.items():
    print(d, "x range", p[:, 0].min().round(), "-", p[:, 0].max().round(), "  y range", p[:, 1].min().round(), "-", p[:, 1].max().round())

## 4. Auto-label every usable val frame

This uses the same procedure as `generate_real_labels` in the pipeline. Each YOLO box goes through its nearest RGB calibration
into the panorama. A footprint becomes *person* if any person box touches it, *other* if any other box does, and
*clear* otherwise. It takes a few minutes on a T4. Nothing is displayed.

In [ ]:
yolo = YOLO(YOLO_WEIGHTS)
FOOTPRINT_PATHS = {c: MplPath(CAM_FOOTPRINTS[c]) for c in SUPERVISED_CAMS}
CLASS_NAMES = {0: "clear", 1: "person", 2: "other"}

def auto_label(img):
    res = yolo(img, verbose=False)[0]
    person = {c: False for c in SUPERVISED_CAMS}
    other = dict(person)
    boxes = []
    for box, cls, conf in zip(res.boxes.xyxy.cpu().numpy(), res.boxes.cls.cpu().numpy(), res.boxes.conf.cpu().numpy()):
        x1, y1, x2, y2 = box
        center = np.array([(x1 + x2) / 2, (y1 + y2) / 2])
        via = min(RGB_CALIBRATIONS, key=lambda c: np.linalg.norm(center - RGB_CALIBRATIONS[c]["centroid"]))
        corners = np.array([[x1, y1, 1], [x2, y1, 1], [x2, y2, 1], [x1, y2, 1]], float)
        proj = (TRANSLATION @ CAM_TRANSFORMS[via] @ RGB_CALIBRATIONS[via]["H"] @ corners.T).T
        proj = proj[:, :2] / proj[:, 2:3]
        pts = list(map(tuple, proj)) + [tuple(proj.mean(0))]
        is_person = int(cls) == COCO_PERSON_CLASS_ID
        hit = [CAM_TO_DIR[c] for c in ACTIVE_CAMERAS if any(FOOTPRINT_PATHS[c].contains_point(p) for p in pts)]
        for c in ACTIVE_CAMERAS:
            if CAM_TO_DIR[c] in hit:
                (person if is_person else other)[c] = True
        boxes.append({"xyxy": [round(float(v), 1) for v in box], "coco_class": int(cls),
                      "class_name": res.names[int(cls)], "conf": round(float(conf), 3), "directions": hit})
    labels = {CAM_TO_DIR[c]: (1 if person[c] else 2 if other[c] else 0) for c in ACTIVE_CAMERAS}
    return labels, boxes

records, BOXES = [], {}
for i, row in enumerate(val_usable):
    part, anchor = row["_part"], int(row["anchor_index"])
    rgb_path = CAPTURE_PARTS[part]["rgb_dir"] / row["rgb_files"]
    img = cv2.imread(str(rgb_path))
    if img is None:
        warnings.warn(f"unreadable RGB frame: {rgb_path}")
        continue
    labels, boxes = auto_label(img)
    BOXES[(part, anchor)] = boxes
    records.append({"part": part, "anchor_index": anchor, "rgb_file": row["rgb_files"],
                    **{f"auto_{d}": labels[d] for d in DIRS}})
    if (i + 1) % 500 == 0:
        print(f"  {i + 1}/{len(val_usable)}")
val_df = pd.DataFrame(records)
print(f"auto-labeled {len(val_df)} val frames")

# aggregate prevalence only: needed later to weight overall agreement back to the val mix
prev = pd.DataFrame({d: val_df[f"auto_{d}"].map(CLASS_NAMES).value_counts(normalize=True) for d in DIRS}).fillna(0)
print("val-set auto-label prevalence (fraction of frames, per direction):")
print(prev.round(3).to_string())

## 5. Sample ~100 frames

Frames are chosen with quotas so the rare classes get enough examples. There are 35 frames where some direction is *person* and 35
where some direction is *other*, each rotated over L/M/R and round-robin over capture sessions. The last 30 are random,
mostly all-clear. No two picks are within `MIN_GAP_ANCHORS` of each other in the same session, to avoid near-duplicates.

In [ ]:
val_df["key"] = list(zip(val_df["part"], val_df["anchor_index"]))
chosen, chosen_keys = [], set()

def too_close(part, anchor):
    return any(p == part and abs(a - anchor) < MIN_GAP_ANCHORS for p, a in chosen_keys)

def take(pool_mask, n, rotate_dirs=None, cls=None):
    parts = sorted(val_df["part"].unique(), key=lambda p: int(re.search(r"\d+", p).group()))
    rng.shuffle(parts)
    got, part_i, misses = 0, 0, 0
    while got < n and misses < len(parts) * 3 + 3:
        mask = pool_mask.copy()
        if rotate_dirs:
            mask &= val_df[f"auto_{rotate_dirs[got % len(rotate_dirs)]}"] == cls
        part = parts[part_i % len(parts)]
        part_i += 1
        cand = val_df[mask & (val_df["part"] == part)]
        cand = [r for r in cand.itertuples() if r.key not in chosen_keys and not too_close(r.part, r.anchor_index)]
        if not cand:
            misses += 1
            continue
        r = rng.choice(cand)
        chosen.append(r.Index)
        chosen_keys.add(r.key)
        got, misses = got + 1, 0
    if got < n:
        warnings.warn(f"only found {got}/{n} frames for this quota")
    return got

everything = pd.Series(True, index=val_df.index)
take(everything, N_PERSON_FRAMES, rotate_dirs=DIRS, cls=1)
take(everything, N_OTHER_FRAMES, rotate_dirs=DIRS, cls=2)
take(everything, N_RANDOM_FRAMES)

rng.shuffle(chosen)                                   # annotation order: quotas fully mixed
sample_df = val_df.loc[chosen].reset_index(drop=True)
sample_df.insert(0, "frame_id", [f"hv_{i:03d}" for i in range(len(sample_df))])

counts = pd.Series([CLASS_NAMES[sample_df[f"auto_{d}"][i]] for d in DIRS for i in range(len(sample_df))]).value_counts()
print(f"{len(sample_df)} frames -> {3 * len(sample_df)} direction labels")
print("directions per auto-label class:", counts.to_dict())
print("frames per session:", sample_df["part"].value_counts().sort_index().to_dict())
low = [c for c in CLASS_NAMES.values() if counts.get(c, 0) < MIN_PER_CLASS]
if low:
    warnings.warn(f"classes below {MIN_PER_CLASS} directions: {low}. Raise the matching quota and re-run this cell")

## 6. Write the annotator folder and the key

In [ ]:
for d in (SAMPLE_DIR, KEY_DIR):
    shutil.rmtree(d, ignore_errors=True)
(SAMPLE_DIR / "images").mkdir(parents=True)
KEY_DIR.mkdir(parents=True)

manifest = {"dataset_id": f"hv_seed{SEED}_n{len(sample_df)}" + ("_SMOKE" if SMOKE_TEST else ""), "frames": []}
for r in sample_df.itertuples():
    img = cv2.imread(str(CAPTURE_PARTS[r.part]["rgb_dir"] / r.rgb_file))
    h, w = img.shape[:2]
    scale = min(1.0, OUT_MAX_WIDTH / w)
    if scale < 1:
        img = cv2.resize(img, (round(w * scale), round(h * scale)), interpolation=cv2.INTER_AREA)
    name = f"images/{r.frame_id}.jpg"
    cv2.imwrite(str(SAMPLE_DIR / name), img, [cv2.IMWRITE_JPEG_QUALITY, 92])
    manifest["frames"].append({
        "frame_id": r.frame_id,
        "image": name,
        "footprints": {d: (RGB_FOOTPRINTS[d] * scale).round(1).tolist() for d in DIRS},
        "directions": DIRS,
    })
(SAMPLE_DIR / "manifest.json").write_text(json.dumps(manifest))

# key: long format, one row per (frame, direction), for the analysis script
key_rows = [{"frame_id": r.frame_id, "part": r.part, "anchor_index": r.anchor_index, "rgb_file": r.rgb_file,
             "direction": d, "auto_label": CLASS_NAMES[getattr(r, f"auto_{d}")]}
            for r in sample_df.itertuples() for d in DIRS]
pd.DataFrame(key_rows).to_csv(KEY_DIR / "auto_labels_key.csv", index=False)
(KEY_DIR / "yolo_boxes.json").write_text(json.dumps(
    {r.frame_id: BOXES[(r.part, r.anchor_index)] for r in sample_df.itertuples()}))
val_df.drop(columns="key").to_csv(KEY_DIR / "val_auto_labels_all.csv", index=False)
(KEY_DIR / "split_info.json").write_text(json.dumps({
    "parts": list(CAPTURE_PARTS), "n_sync_rows": len(all_rows), "n_train": len(train_rows),
    "n_val": len(val_rows), "n_val_usable": len(val_usable), "seed": SEED,
    "quotas": [N_PERSON_FRAMES, N_OTHER_FRAMES, N_RANDOM_FRAMES], "min_gap_anchors": MIN_GAP_ANCHORS}, indent=1))

for d in (SAMPLE_DIR, KEY_DIR):
    with zipfile.ZipFile(OUT / f"{d.name}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(d.rglob("*")):
            zf.write(f, f.relative_to(d.parent))
    print(f"wrote {OUT / (d.name + '.zip')}  ({(OUT / (d.name + '.zip')).stat().st_size / 1e6:.1f} MB)")

## 7. Sanity check: footprints only, no labels

Check that the L/M/R footprints sit where the thermal cameras look and are in the right left-to-right order.
If they look mirrored or offset, stop and fix that before anyone labels.

In [ ]:
COLORS = {"L": "#2563eb", "M": "#0d9488", "R": "#9333ea"}
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, fr in zip(axes, manifest["frames"][:3]):
    ax.imshow(cv2.cvtColor(cv2.imread(str(SAMPLE_DIR / fr["image"])), cv2.COLOR_BGR2RGB))
    for d, poly in fr["footprints"].items():
        ax.add_patch(plt.Polygon(poly, fill=False, lw=2, ec=COLORS[d]))
        ax.text(*np.mean(poly, 0), d, color="white", fontsize=14, weight="bold", ha="center",
                bbox=dict(fc=COLORS[d], ec="none", alpha=.8))
    ax.set_title(fr["frame_id"]); ax.axis("off")
plt.tight_layout(); plt.show()